In [1]:
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [3]:
import sys
from PIL import Image
import numpy as np
import openslide
from tqdm import tqdm
import pickle

In [4]:
sys.path.append(os.path.join(os.getcwd(), 'histocartography'))
from histocartography.preprocessing import NucleiExtractor, DeepFeatureExtractor, KNNGraphBuilder
from histocartography.visualization import OverlayGraphVisualization, InstanceImageVisualization

/data/mn27889/miniconda3/envs/pbt-histo/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initiating all the required modules

In [5]:
nuclei_detector = NucleiExtractor()
feature_extractor = DeepFeatureExtractor(architecture='resnet34', patch_size=224, resize_size=224)
knn_graph_builder = KNNGraphBuilder(k=5, thresh=50, add_loc_feats=True)

File already downloaded.


/data/mn27889/pbt-histocartography/histocartography/histocartography/preprocessing/nuclei_extraction.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model = torch.

Specifying the directories of Raw WSIs

In [6]:
# raw_wsi_dir = "/data/jy26539/data/wsi_raw"
raw_wsi_dir = "/data/mn27889/pbt-histocartography/tcga_download/tcga_gbm_lgg_pedi"
wsi_names = os.listdir(raw_wsi_dir)
print(f"Found {len(wsi_names)} WSIs in {raw_wsi_dir}.")

Found 283 WSIs in /data/mn27889/pbt-histocartography/tcga_download/tcga_gbm_lgg_pedi.


In [7]:
wsi_resolution_level = 1 # Change this to adjust the resolution level for processing (0 is highest resolution)
wsi_features_dir = f"wsi_features_tcga_resolution_{wsi_resolution_level}"
wsi_cell_graph_dir = f"wsi_cell_graphs_tcga_resolution_{wsi_resolution_level}"
wsi_cell_graph_viz_dir = f"wsi_cell_graphs_tcga_viz_resolution_{wsi_resolution_level}"
os.makedirs(wsi_features_dir, exist_ok=True)
os.makedirs(wsi_cell_graph_dir, exist_ok=True)
os.makedirs(wsi_cell_graph_viz_dir, exist_ok=True)

Generating Nuclei, Features and Graphs for all WSIs

In [ ]:
for i in tqdm(range(0, len(wsi_names)), desc="Processing"):
    wsi_name = wsi_names[i]
    print(f"\nProcessing {wsi_name} ({i+1}/{len(wsi_names)})...")
    wsi_path = os.path.join(raw_wsi_dir, wsi_name)
    wsi_cell_graph_viz_path = os.path.join(wsi_cell_graph_viz_dir, f"{i}_cell_graph_viz_{wsi_name}.png")
    try:
        slide = openslide.OpenSlide(wsi_path)
        svs_image = slide.read_region((0, 0), wsi_resolution_level, slide.level_dimensions[wsi_resolution_level]).convert('RGB')
        svs_image_np = np.array(svs_image)
        
        nuclei_map, nuclei_centers = nuclei_detector.process(svs_image_np)
        features = feature_extractor.process(svs_image_np, nuclei_map)
        with open(os.path.join(wsi_features_dir, f"{i}_features_{wsi_name}.pkl"), 'wb') as f:
            pickle.dump(features.detach().cpu().numpy(), f)  # Save features for later use
        
        if len(nuclei_centers) > 5:
            cell_graph = knn_graph_builder.process(nuclei_map, features)
            with open(os.path.join(wsi_cell_graph_dir, f"{i}_cell_graph_{wsi_name}.pkl"), 'wb') as f:
                pickle.dump(cell_graph, f)  # Save cell graph for later use
            
            # visualizer = OverlayGraphVisualization(instance_visualizer=InstanceImageVisualization(instance_style="filled+outline"))
            # viz_cg = visualizer.process(canvas=svs_image_np, graph=cell_graph, instance_map=nuclei_map)
            # viz_cg.save(wsi_cell_graph_viz_path)
            # print(f"Saved graph visualization to {wsi_cell_graph_viz_path}")
            print(f"Processed {wsi_name}")
        else:
            print(f"Less than 5 nuclei detected in {wsi_name}. Skipping graph construction and visualization.")

        slide.close()
    except Exception as e:
        print(f"!!!!!!!!!!!Error processing {wsi_name} ({i+1}/{len(wsi_names)}): {e}!!!!!!!!!")

Combining the features and cell graphs of all WSIs into a single dictionary

In [8]:
features_dict = {}
cell_graph_dict = {}

for i in tqdm(range(0, len(wsi_names)), desc="Processing"):
    wsi_name = wsi_names[i]
    
    # Combining the features
    feature_path = os.path.join(wsi_features_dir, f"{i}_features_{wsi_name}.pkl")
    if os.path.exists(feature_path):
        with open(feature_path, 'rb') as f:
            features_dict[wsi_name] = pickle.load(f)
    else:
        print(f"Feature file not found for {wsi_name}. Skipping.")

    # Combining the Cell Graphs
    cell_graph_path = os.path.join(wsi_cell_graph_dir, f"{i}_cell_graph_{wsi_name}.pkl")
    if os.path.exists(cell_graph_path):
        with open(cell_graph_path, 'rb') as f:
            cell_graph_dict[wsi_name] = pickle.load(f)
    else:
        print(f"Cell graph file not found for {wsi_name}. Skipping.")

with open(f'wsi_features_tcga_resolution_{wsi_resolution_level}.pkl', 'wb') as f:
    pickle.dump(features_dict, f)

with open(f'wsi_cell_graphs_tcga_resolution_{wsi_resolution_level}.pkl', 'wb') as f:
    pickle.dump(cell_graph_dict, f)

Processing:   0%|          | 0/283 [00:00<?, ?it/s]/data/mn27889/miniconda3/envs/pbt-histo/lib/python3.9/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

Feature file not found for TCGA-HT-7858-01Z-00-DX4.7F6E369E-6CA5-4C61-A0B6-75C3EDDED6A9.svs. Skipping.
Cell graph file not found for TCGA-HT-7858-01Z-00-DX4.7F6E369E-6CA5-4C61-A0B6-75C3EDDED6A9.svs. Skipping.
Feature file not found for TCGA-FG-A4MT-02Z-00-DX5.BC137126-2035-4629-BCCF-EDDD39CC7178.svs. Skipping.
Cell graph file not found for TCGA-FG-A4MT-02Z-00-DX5.BC137126-2035-4629-BCCF-EDDD39CC7178.svs. Skipping.
Feature file not found for TCGA-HT-7610-01Z-00-DX2.FA27FEBA-2400-451A-9E7E-492D150AAAA7.svs. Skipping.
Cell graph file not found for TCGA-HT-7610-01Z-00-DX2.FA27FEBA-2400-451A-9E7E-492D150AAAA7.svs. Skipping.
Feature file not found for TCGA-HT-7473-01Z-00-DX6.76D0E72C-D2EA-4B6A-B43B-28BF23E7FD3B.svs. Skipping.
Cell graph file not found for TCGA-HT-7473-01Z-00-DX6.76D0E72C-D2EA-4B6A-B43B-28BF23E7FD3B.svs. Skipping.
Feature file not found for TCGA-P5-A5ET-01A-01-TSA.3733C4A4-9D3E-4A02-BE96-CF0C7E4664B1.svs. Skipping.
Cell graph file not found for TCGA-P5-A5ET-01A-01-TSA.3733C4A

Read the feature_dict and cell_graph_dict

In [9]:
with open(f'wsi_features_tcga_resolution_{wsi_resolution_level}.pkl', 'rb') as f:
    features_dict = pickle.load(f)

with open(f'wsi_cell_graphs_tcga_resolution_{wsi_resolution_level}.pkl', 'rb') as f:
    cell_graph_dict = pickle.load(f)